<a href="https://colab.research.google.com/github/kamujunaganandini-oss/Explainable-Anomaly-Intelligence/blob/main/Named_Entity_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [13]:
import pandas as pd
import numpy as np

## Data Loading

In this step, we load the dataset using pandas and understand its structure.

The dataset contains:
- Sentence #: Sentence identifier
- Word: Individual word in the sentence
- POS: Part-of-speech tag (not used in this project)
- Tag: Named Entity label (IOB format)

We will first inspect:
- First few rows
- Dataset shape
- Column names

In [4]:

# Load dataset
df = pd.read_csv("ner_dataset.csv", encoding="latin1")

# Show first few rows
print("First 5 rows:")
print(df.head())

# Dataset shape
print("\nDataset Shape:", df.shape)

# Column names
print("\nColumns:", df.columns)

First 5 rows:
    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O

Dataset Shape: (1048575, 4)

Columns: Index(['Sentence #', 'Word', 'POS', 'Tag'], dtype='object')


## Handling Missing Values

The dataset contains missing values in the "Sentence #" column.

To fix this, we use **forward fill (ffill)**:
- It propagates the last valid value forward
- Ensures every word is correctly assigned to a sentence

This step is important for grouping words into complete sentences.

In [5]:
# Fill missing values
df = df.fillna(method='ffill')

print("Missing values after fill:")
print(df.isnull().sum())

/tmp/ipykernel_1338/3008265500.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')


Missing values after fill:
Sentence #    0
Word          0
POS           0
Tag           0
dtype: int64


##Converting Words into Sentences

Currently, the dataset is at **word level**.

We convert it into **sentence-level format** by grouping:
- Words → Input (X)
- Tags → Output (y)

Final structure:
- X = list of sentences (list of words)
- y = corresponding list of tag sequences

This format is required for training and evaluation.

In [7]:
# Group data by Sentence #
grouped = df.groupby("Sentence #")

# Create sentences (X) and tags (y)
X = []
y = []

for _, group in grouped:
    words = list(group["Word"])
    tags = list(group["Tag"])

    X.append(words)
    y.append(tags)

# Check one example
print("Sample Sentence:")
print(X[0])

print("\nSample Tags:")
print(y[0])

Sample Sentence:
['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.']

Sample Tags:
['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O']


##Train / Validation / Test Split

We split the dataset into:
- Training set (70%) → Used to build the model
- Validation set (10%) → Used for tuning
- Test set (20%) → Used for final evaluation

We use `train_test_split` with a fixed `random_state` to ensure reproducibility.

In [8]:
from sklearn.model_selection import train_test_split

# First split: Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Second split: Validation (10%) and Test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=2/3, random_state=42
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

Train size: 33571
Validation size: 4796
Test size: 9592


##Baseline Model: Most Frequent Tag per Word

I built a simple baseline model:

Idea:
For each word, assign the **most frequent tag** seen in training data.

Example:
- "India" → B-LOC (if most common)
- "John" → B-PER

For unseen words:
- Assign default tag → "O"

This model is simple but provides a good starting point for evaluation.

In [9]:
from collections import defaultdict, Counter

# Dictionary: word -> most frequent tag
word_tag_dict = {}

# Temporary dictionary: word -> list of tags
word_tag_counts = defaultdict(list)

# Collect tags for each word
for sentence, tags in zip(X_train, y_train):
    for word, tag in zip(sentence, tags):
        word_tag_counts[word].append(tag)

# Find most common tag for each word
for word, tag_list in word_tag_counts.items():
    most_common_tag = Counter(tag_list).most_common(1)[0][0]
    word_tag_dict[word] = most_common_tag

print("Sample word-tag mappings:")
for i, (k, v) in enumerate(word_tag_dict.items()):
    print(k, "->", v)
    if i == 5:
        break

Sample word-tag mappings:
Mousavi -> I-per
's -> O
website -> O
also -> O
quotes -> O
him -> O


##Prediction Function

We define a function that:
- Takes a sentence (list of words)
- Returns predicted NER tags

Logic:
- If word seen during training → use learned tag
- If unseen → assign "O"

This function will be used for evaluation.

In [10]:
def predict_sentence(sentence, word_tag_dict):
    predictions = []

    for word in sentence:
        if word in word_tag_dict:
            predictions.append(word_tag_dict[word])
        else:
            predictions.append("O")  # Default for unseen words

    return predictions

# Test prediction
print("Sentence:", X_test[0])
print("Predicted:", predict_sentence(X_test[0], word_tag_dict))

Sentence: ['From', '2004', 'to', '2007', ',', 'the', 'economy', 'grew', 'about', '10', '%', 'per', 'year', ',', 'driven', 'largely', 'by', 'an', 'expansion', 'in', 'the', 'garment', 'sector', ',', 'construction', ',', 'agriculture', ',', 'and', 'tourism', '.']
Predicted: ['O', 'B-tim', 'O', 'I-tim', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


##Model Evaluation

We evaluate the model on the test set using:

classification_report from sklearn

Steps:
- Predict tags for each sentence
- Flatten predictions and true labels
- Compare using Precision, Recall, and F1-score

This helps us understand model performance across different entity types.

In [11]:
from sklearn.metrics import classification_report

# Flatten lists
y_true = []
y_pred = []

for sentence, true_tags in zip(X_test, y_test):
    pred_tags = predict_sentence(sentence, word_tag_dict)

    y_true.extend(true_tags)
    y_pred.extend(pred_tags)

# Generate report
print("Classification Report:\n")
print(classification_report(y_true, y_pred))

Classification Report:

              precision    recall  f1-score   support

       B-art       0.25      0.08      0.13        95
       B-eve       0.50      0.28      0.36        58
       B-geo       0.78      0.86      0.82      7515
       B-gpe       0.94      0.93      0.93      3135
       B-nat       0.29      0.30      0.29        37
       B-org       0.66      0.51      0.58      4167
       B-per       0.77      0.66      0.71      3365
       B-tim       0.87      0.77      0.82      4007
       I-art       0.08      0.01      0.02        90
       I-eve       0.37      0.15      0.21        47
       I-geo       0.73      0.60      0.66      1437
       I-gpe       0.48      0.38      0.43        26
       I-nat       0.00      0.00      0.00        15
       I-org       0.70      0.54      0.61      3466
       I-per       0.73      0.68      0.70      3382
       I-tim       0.59      0.13      0.21      1262
           O       0.97      0.99      0.98    177165

  

##Results & Sample Predictions

Results:
- Model performance (classification report)
- Example sentences with:
  - Actual tags
  - Predicted tags

In [12]:
for i in range(3):
    print(f"\nExample {i+1}:")
    print("Sentence:", X_test[i])
    print("True Tags:", y_test[i])
    print("Predicted Tags:", predict_sentence(X_test[i], word_tag_dict))


Example 1:
Sentence: ['From', '2004', 'to', '2007', ',', 'the', 'economy', 'grew', 'about', '10', '%', 'per', 'year', ',', 'driven', 'largely', 'by', 'an', 'expansion', 'in', 'the', 'garment', 'sector', ',', 'construction', ',', 'agriculture', ',', 'and', 'tourism', '.']
True Tags: ['B-tim', 'I-tim', 'I-tim', 'I-tim', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Predicted Tags: ['O', 'B-tim', 'O', 'I-tim', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Example 2:
Sentence: ['Earlier', 'this', 'week', ',', 'the', 'African', 'Union', 'dispatched', 'former', 'South', 'African', 'president', 'Thabo', 'Mbeki', 'to', 'seek', 'a', 'solution', ',', 'but', 'after', 'two', 'days', 'of', 'talks', ',', 'the', 'situation', 'was', 'not', 'resolved', '.']
True Tags: ['O', 'O', 'O', 'O', 'O', 'B-geo', 'I-geo', 'O', 'O', 'B-g

##Limitations of Baseline Model

This baseline model has several limitations:

1. No context awareness  
   - Each word is treated independently

2. Cannot handle unseen words well  
   - Defaults to "O"

3. Ambiguity problem  
   - Same word may have different meanings

4. No sequence understanding  
   - Does not capture tag dependencies (B → I)

These limitations motivate the need for more advanced models.

##Improvement: CRF Model

A better approach is to use **Conditional Random Fields (CRF)**.

CRF improves performance by:
- Considering the entire sentence
- Learning relationships between neighboring tags
- Ensing valid tag sequences

This helps in capturing context and improving accuracy.

###Feature Engineering

CRF does not work directly on raw words.  
We convert each word into a set of features.

Features used:
- Word itself
- Lowercase form
- Prefixes and suffixes
- Capitalization information
- Previous word (context)
- Next word (context)

These features help the model understand both the word and its surrounding context.

In [14]:
pip install sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.5 MB/s eta 0:00:00


###Converting Sentences to Features

We convert each sentence into a list of feature dictionaries.

Each word in the sentence is transformed into features using the `word2features` function.

Final structure:
- Input: List of sentences
- Output: List of feature sequences (one per sentence)

This format is required for training the CRF model.

In [17]:
#reusing the same features
def word2features(sentence, i):
    word = sentence[i]

    features = {
        'word': word,
        'lower': word.lower(),
        'is_upper': word.isupper(),
        'is_title': word.istitle(),
        'suffix': word[-2:],
        'prefix': word[:2],
    }

    if i > 0:
        features['prev_word'] = sentence[i-1]
    else:
        features['BOS'] = True

    if i < len(sentence) - 1:
        features['next_word'] = sentence[i+1]
    else:
        features['EOS'] = True

    return features


def sentence_to_features(sentence):
    return [word2features(sentence, i) for i in range(len(sentence))]

###Preparing Training Data

Unlike traditional machine learning models, CRF works on sequence data.

Important:
- We do NOT flatten the data
- Each sentence remains a sequence of feature dictionaries
- Tags are also kept as sequences

This allows the model to learn dependencies between tags.

In [18]:
X_train_crf = [sentence_to_features(s) for s in X_train]
X_test_crf  = [sentence_to_features(s) for s in X_test]

###Training the CRF Model

We train the CRF model using:
- L-BFGS optimization algorithm
- Limited number of iterations for efficiency

The model learns:
- Relationship between features and tags
- Transition probabilities between tags (e.g., B-LOC → I-LOC)

This helps in generating valid and consistent tag sequences.

In [19]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=100
)

crf.fit(X_train_crf, y_train)

CRF(algorithm='lbfgs', max_iterations=100)

###Making Predictions

Once trained, the CRF model predicts tags for entire sentences.

Unlike the baseline model:
- Predictions are made considering the full sequence
- Tag dependencies are maintained

This results in more accurate and structured outputs.

In [20]:
y_pred_crf = crf.predict(X_test_crf)

###Model Evaluation

We evaluate the CRF model using:
- Precision
- Recall
- F1-score

Steps:
- Predict tags for test sentences
- Flatten predictions and true labels
- Generate classification report

This allows us to compare performance with the baseline model.

In [21]:
from sklearn.metrics import classification_report

y_true = [tag for sent in y_test for tag in sent]
y_pred = [tag for sent in y_pred_crf for tag in sent]

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

       B-art       0.33      0.03      0.06        95
       B-eve       0.63      0.38      0.47        58
       B-geo       0.86      0.91      0.88      7515
       B-gpe       0.95      0.92      0.94      3135
       B-nat       0.47      0.19      0.27        37
       B-org       0.81      0.71      0.75      4167
       B-per       0.84      0.80      0.82      3365
       B-tim       0.93      0.87      0.90      4007
       I-art       0.33      0.03      0.06        90
       I-eve       0.48      0.34      0.40        47
       I-geo       0.82      0.82      0.82      1437
       I-gpe       1.00      0.46      0.63        26
       I-nat       0.80      0.27      0.40        15
       I-org       0.78      0.81      0.79      3466
       I-per       0.84      0.90      0.87      3382
       I-tim       0.84      0.76      0.80      1262
           O       0.99      0.99      0.99    177165

    accuracy              

###Why CRF Performs Better

CRF improves over the baseline model because:

1. Context Awareness  
   - Uses previous and next words

2. Sequence Modeling  
   - Learns valid tag transitions

3. Better Generalization  
   - Uses features instead of memorization

4. Structured Predictions  
   - Ensures consistent tag sequences

Overall, CRF provides more accurate and reliable NER predictions.